# Prep route and stop gdf
1. only subset for operators by region
2. do aggregation?
3. subset to special routes
   if this is skipped, after step 2, can go straight to regression?

* https://xyzservices.readthedocs.io/en/stable/introduction.html

In [1]:
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars

credentials, _ = google.auth.default()
GCS_FILE_PATH = wc_vars.GCS_FILE_PATH

In [2]:
route_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}route_summary_sofi.parquet",
    storage_options = {"token": credentials}
)

In [8]:
import E1_new_prep as E1
sofi_trips = E1.prep_fct_daily_schedule_rt_route_direction_summary(
    wc_vars.event_name,
    wc_vars.socal_names,
    wc_vars.sofi_match_times,
)

In [13]:
sofi_trips.dtypes

shape_array_key              object
geometry                   geometry
service_date         datetime64[ns]
schedule_name                object
feed_key                     object
route_id                     object
route_id_cleaned             object
route_name                   object
direction_id                  int64
route_type                   object
shape_id                     object
n_trips                       int64
num_stop_times                int64
event_day                      bool
day_type                     object
event_time_of_day            object
dtype: object

In [12]:
routes_near_poi = E1.import_routes_near_poi(wc_vars.event_name, "sofi").query(
    'is_special_route==True')

In [17]:
test2 = pd.merge(
    sofi_trips,
    routes_near_poi,
    on = ["feed_key", "schedule_name", "route_id", "direction_id", "route_name"],
    how = "inner"
)

In [20]:
import C4_event_helpers as C4
daily_trips_by_route = C4.aggregate_daily_trips(
    test2, ["service_date", "schedule_name", "route_name"]
)

In [24]:

import chart_utils

chart_utils.trip_chart_with_event_dates(
    daily_trips_by_route, wc_vars.sofi_dates, color_col="route_name"
).properties(
    title= "Daily Trips by Route during World Cup",
    width=500, height=300
)

alt.LayerChart(...)

In [3]:
stop_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}stop_summary_sofi.parquet",
    storage_options = {"token": credentials}
)

In [ ]:
from dotenv import dotenv_values

CARTO_TILE_URL = dotenv_values("_env")["CARTO_TILE_URL"]

In [ ]:
sofi_gdf.explore(
    "is_near",
    tiles = CARTO_TILE_URL,
    attr="(C) OpenStreetMap contributors (C) CARTO"
)